In [9]:
import json

json_path = '../dataset/original/inference_train.json'
output_path = '../dataset/ellipsis_recovered/train.json'

with open(json_path, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

print(len(dataset))

758


In [10]:
def format_dialogue(item):
    """conversation 배열을 '화자 1: 말\n화자 2: 말\n...' 형식의 문자열로 변환"""
    conv = item["input"]["conversation"]
    lines = []
    for turn in conv:
        sp = turn.get("speaker")
        utt = turn.get("utterance", "")
        # "화자 1: "처럼 표현 (스페이스 포함)
        lines.append(f"화자 {sp}: {utt}")
    return "\n".join(lines)


In [11]:
import re, os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.messages import HumanMessage

from dotenv import load_dotenv
load_dotenv()
#os.environ['OPENAI_API_KEY'] = "YOUR_API_KEY"

True

In [30]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    request_timeout=60,
    api_key=os.environ["OPENAI_API_KEY"]
)

system_prompt = """
You are an assistant specialized in restoring omitted components in Korean dialogue lines and merging consecutive utterances by the same speaker.

*Core Rules (Crucial)*
1. You will receive a dialogue as plain text. Output only the dialogue, with all minimally necessary restorations inside square brackets [ ].
2. Merge consecutive lines from the same speaker into a single line. Keep the original order of phrases while merging. Separate clauses with natural spacing and punctuation.
3. Correct obvious typos, spacing, and malformed colloquialisms when it does not change meaning (e.g., “해소”→“해서”, “가티”→“같이”).
4. Do not over-restore. Preserve only essential sentence elements (subjects, objects, particles, predicates, key adjuncts). Do not resurrect trivial or redundant omissions (fillers, repeated function words, obvious ellipses) if they are unnecessary for clarity.
5. Do not change meaning, tense, or referents (“that/then/there”). If uncertain, prefer minimal restoration.
6. If multiple insertions are needed in a line, place each restored piece exactly where it belongs, each inside its own [ ].

*Processing Guidelines (for internal reasoning only)*
1. Use discourse context (topic continuity, before/after turns) to infer omitted elements, then restore only what is necessary for grammaticality and clarity using [ ].
2. Normalize colloquial contractions if meaning is unchanged (e.g., “했지” can remain as-is; add particles like “[가]”, “[을]” only when needed).
3. When merging same-speaker lines, keep laughter/interjections/emojis inline if they contribute to tone, but avoid duplicating them unnecessarily.

*Output Format*
1. Output only the dialogue, preserving speaker labels (화자 1:, 화자 2: …).
2. After merging, each speaker block should be a single line per contiguous turn (i.e., no duplicate consecutive lines for the same speaker).
3. All restorations must be shown inside square brackets [ ].
4. Do not include any explanations, headers, or metadata—only the merged, restored dialogue.

---

**Few-shot examples (mimic exactly this format)**

Example 1:
Input:
화자 2: 진짜 신의 한수
화자 1: 이사하자마자 비 많이 와서 베란다 물 많이 새는 거 알았잖아
화자 2: 글치 계속 해떴으면 몰랐겠지
화자 1: 그 때 물새는 거 알고 코킹작업해소 다행이다
화자 2: ㅇㅇ 안그랬으면 오늘처럼 비 많이 내리는 날 물바다됐을거야
화자 1: 요 아래 씽크홀 공사하던데 괜찮을라나
화자 2: 그러게 저번에도 비 많이 와서 땅꺼진 건데 큰일이네
화자 1: 하수도 공사도 같이 하더만 물 안빠져서
화자 2: 새로 지은 곳인데도 그러네
화자 1: 부실공사지 뭐
화자 2: 비 많이 올 때는 그쪽으로 다니지 말아야겠다
화자 1: ㅇㅇ 조심해
화자 1: 저번에 지나가다 보니 좀 무섭더라
화자 2: 나도 봤는데 씽크홀 크기가 엄청나더라
화자 1: 오늘 비가 엄청 많이 내리네

Output:
화자 2: 진짜 [이사한 게] 신의 한수[였어]
화자 1: [우리가] 이사하자마자 비[가] 많이 와서 베란다[에] 물[이] 많이 새는 거 알았잖아
화자 2: 글치 [그걸] 계속 [해가] 떴으면 [우리는] 몰랐겠지
화자 1: 그 때 [베란다에] 물 새는 거 알고 코킹 작업해[서] 다행이다
화자 2: ㅇㅇ 안 그랬으면 오늘처럼 비[가] 많이 내리는 날 [집이] 물바다 됐을 거야
화자 1: 요 아래 씽크홀 공사하던데 [그게] 괜찮을라나
화자 2: 그러게 저번에도 비[가] 많이 와서 땅[이] 꺼진 건데 큰일이네
화자 1: 하수도 공사도 같이 하더만 물[이] 안 빠져서
화자 2: [거기가] 새로 지은 곳인데도 그러네
화자 1: [그건] 부실 공사[지] 뭐
화자 2: 비[가] 많이 올 때는 [우리는] 그쪽으로 다니지 말아야겠다
화자 1: ㅇㅇ 조심해 저번에 [거길] 지나가다 보니 좀 무섭더라
화자 2: 나도 봤는데 [그] 씽크홀[의] 크기[가] 엄청나더라
화자 1: 오늘 비[가] 엄청 많이 내리네

Example 2
Input: 
화자 2: 어제다친발목에
화자 2: 파스를계속붙엿더니
화자 2: 따갑습니다요..
화자 2: ㄱㅋㄱㅋㄱㅋ
화자 1: ㅋㅋㅋ아
화자 1: 친구들이 아이폰인게 계속 슬프네요
화자 1: ㅋㅋㅋname2님이랑 가티 하차해야것어유
화자 1: ㅋㅋㅋㅋ
화자 1: 파스말구
화자 1: 온찜질!!
화자 2: 아 온찜질인가여..!!!!!!
화자 1: 수건물에적셔서 전자렌지 돌리세욥!!!!!
화자 2: 악저랑하차하신다니익!!!!!!
화자 2: 오 팁감사합니다 진짜하구자야겟어요
화자 2: ㅠㅡㅠ
화자 1: ㅎㅎ강추!!!!
화자 2: 나이들수록
화자 2: 다친곳 또 다치구
화자 2: 이런게많아지는거같아요
화자 1: 특히 발목은
화자 1: 더그래요ㅜㅜ
화자 2: 안그랬는데갑자기몇달전에
화자 2: 스피드민턴친다고설치다가 한번다치고나서
화자 2: 그때부터ㅋㅋㄲㅋ
화자 1: 아이공
화자 1: 온찜질 계속 해주세유
화자 2: 네 ㅠㅡㅠ  아직도 냉찜질온찜질 헷갈립니다..ㅋㅋㅋㅋ어디에뭐가좋은지
화자 1: ㅋㅋ발목을 애껴주세요
화자 1: ㅋㅋㅋㅋ
화자 2: 그러게요소중히다뤄야하는데ㅜㅜ

Output:
화자 2: 어제 [다친] 발목[이] 파스를 계속 붙였더니 따갑습니다[요].. ㄱㅋㄱㅋㄱㅋ
화자 1: ㅋㅋㅋ 아 친구들이 아이폰[을] [쓰고] 있는 게 계속 슬프네요 ㅋㅋㅋ name2님이랑 [같이] 하차해야겠어유 ㅋㅋㅋㅋ 파스 말고 온찜질!!
화자 2: 아, 온찜질인가여..!!!!!!
화자 1: 수건[을] 물에 적셔서 전자렌지 돌리세요!!!!!
화자 2: 악, 저랑 하차하신다니!!!!!! 오, 팁 감사합니다. 진짜 하고 자야겟어요 ㅠㅡㅠ
화자 1: ㅎㅎ 강추!!!!
화자 2: 나이 들수록 다친 곳 또 다치고 이런 게 많아지는 거 같아요
화자 1: 특히 발목은 더 그래요ㅜㅜ
화자 2: 안 그랬는데 갑자기 몇 달 전에 스피드민턴[을] 친다고 설치다가 한번 다치고 나서 그때부터 [다친 곳을 또 다치네요]  ㅋㅋㄲㅋ
화자 1: 아이공 온찜질 [발목에] 계속 해주세유
화자 2: 네 ㅠㅡㅠ  아직도 냉찜질온찜질 [어디에 어떤게 좋은지] 헷갈립니다..ㅋㅋㅋㅋ어디에 [냉온 찜질중에] 뭐가좋은지
화자 1: ㅋㅋ발목을 애껴주세요 ㅋㅋㅋㅋ
화자 2: 그러게요 [발목을] 소중히다뤄야하는데ㅜㅜ
"""

integrated_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt.strip()),
    ("human", "<Actual Dialouge to Process>\n{actual_dialouge}")
])


In [40]:
import time
from typing import Optional

def recover_ellipsis(dialogue_text: str, max_retries: int = 3, backoff: float = 2.0) -> Optional[str]:
    for attempt in range(1, max_retries + 1):
        try:
            chain = integrated_prompt | llm
            resp = chain.invoke({"actual_dialouge": dialogue_text})
            content = resp.content.strip()
            # 산출물 형식 검증(최소: '화자 '로 시작하는 라인 존재)
            if "화자 " in content:
                return content
            else:
                # 형식이 기대와 다르면 재시도
                raise ValueError("Unexpected output format.")
        except Exception as e:
            if attempt == max_retries:
                print(f"[ERROR] Recover failed after {attempt} attempts: {e}")
                return None
            time.sleep(backoff * attempt)

# 테스트 (한 건만)
test_dialogue = format_dialogue(dataset[120])
test_out = recover_ellipsis(test_dialogue)

print("<원본>")
print(test_dialogue)
print("<복구>")
print(test_out if test_out else "생략 복구 실패")

<원본>
화자 2: 오 정말? 무슨 일인데???
화자 1: 아 무슨 일인지 그동안 모랄ㅆ는데 이번에 일을줬어. 데이터 수집하고 웹사이트에 올리는거
화자 2: 빅데이터 관련 업무인가??낯선분야라 신기하다 일한지는 얼마나 됐어ㅓ??
화자 1: 빅데이터랑은 좀 다르더라 거긴 전문적이고 여긴 좀 덜 전문적 ㅋㅋ 이제 이주됐어
화자 2: 그렇구나 지금 한창 적응하고 잇겠네~ 일은 할 만해??
화자 1: 그동안 할만했는데 이제어려워졌었어..ㅠ
화자 2: 헉ㅠㅠ 앞날이 많이 힘들겠구나 머리가 아프겠는걸
화자 1: 그니까 어떠케 ㅠ 나 너무 바보같아서 아무것도 못하겟다능
화자 2: 아니야 처음하면 다들 똑같이 헤매지ㅠㅠ
화자 1: 마자..잘하기보다도 그냥..적당히만 하기라도 바랄뿐
화자 1: ㅠㅠ 난생 처음 보는 분야라너무 생소해..
화자 2: 너가 잘하길 응원할게
화자 2: 같이 일하는 동료들은 많아?
화자 1: 같이 5명 일해 나까지 5명  ㅎㅎㅎ...ㅎㅎㅎ...새롭다 새로워
화자 1: 너는 요새 일 괜찮아? 요새는ㅇ ㅓ때?
화자 2: 나는 늘 똑같지 뭐ㅎㅎ
<복구>
화자 2: 오 정말? 무슨 일인데???
화자 1: 아, 무슨 일인지 그동안 몰랐는데 이번에 일을 줬어. 데이터 수집하고 웹사이트에 올리는 거
화자 2: 빅데이터 관련 업무인가?? 낯선 분야라 신기하다. 일한 지는 얼마나 됐어??
화자 1: 빅데이터랑은 좀 다르더라. 거긴 전문적이고 여긴 좀 덜 전문적 ㅋㅋ 이제 이주[일] 됐어
화자 2: 그렇구나. 지금 한창 적응하고 있겠네~ 일은 할 만해??
화자 1: 그동안 할 만했는데 이제 어려워졌어..ㅠ
화자 2: 헉ㅠㅠ 앞날이 많이 힘들겠구나. 머리가 아프겠는걸
화자 1: 그니까, 어떡해 ㅠ 나 너무 바보 같아서 아무것도 못하겠다는[거야]
화자 2: 아니야, 처음 하면 다들 똑같이 헤매지ㅠㅠ
화자 1: 맞아.. 잘하기보다도 그냥.. 적당히만 하기라도 바랄 뿐 ㅠㅠ 난생 처음 보는 분야라 너무 생소해..
화자 2: 너가 잘하길 응원할게. 같이 일하는 동료들은 많아?


In [17]:
def map_output_to_choice(output_str: str) -> str:
    """
    'inference_1' -> 'A', 'inference_2' -> 'B', 'inference_3' -> 'C'
    """
    m = re.search(r'inference_(\d+)', output_str)
    if not m:
        return ""  # 또는 None
    idx = int(m.group(1))
    return {1: "A", 2: "B", 3: "C"}.get(idx, "")

def build_question_with_options(item) -> str:
    """
    기존 question에 선지 3개를 덧붙여 최종 question 문자열 생성
    """
    q0 = item["input"]["question"].strip()
    inf1 = item["input"].get("inference_1", "").strip()
    inf2 = item["input"].get("inference_2", "").strip()
    inf3 = item["input"].get("inference_3", "").strip()

    # 사용자 예시 형식에 맞춤
    tail = (
        "아래의 선지 중, 해당 사건이 성립하려면 어떤 상태가 먼저 갖춰져야 하나요?\n\n"
        "[선지]\n"
        f"A. {inf1}\n"
        f"B. {inf2}\n"
        f"C. {inf3}"
    )
    return f"{q0}\n\n{tail}"


In [ ]:
from tqdm import tqdm

converted = []
for item in tqdm(dataset, total=len(dataset)):
    _id = item["id"]
    dialogue_plain = format_dialogue(item)

    # 1) GPT로 생략 복구
    dialogue_recovered = recover_ellipsis(dialogue_plain)
    if dialogue_recovered is None:
        # 실패 시 원본 포맷을 그냥 사용(혹은 건너뛰기)
        dialogue_recovered = dialogue_plain

    # 2) 질문/선지
    question_final = build_question_with_options(item)

    # 3) 정답 매핑
    answer_letter = map_output_to_choice(item.get("output", ""))

    converted.append({
        "id": _id,
        "dialouge": dialogue_recovered,   # (요청대로 'dialouge' 키 사용)
        "question": question_final,
        "answer": answer_letter
    })

# 저장
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(converted, f, ensure_ascii=False, indent=2)

len(converted), output_path